[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/04_webscraping/14_web_scraping_fundamentals.ipynb)

# 📓 Notebook 14 — Web Scraping Fundamentals

> **Module:** Web Scraping (Module 4) · **Estimated time:** ~70 minutes · **Difficulty:** Intermediate

Most of the world's data has **no API**. It sits in HTML pages built for human eyes — product listings, catalogues, tables, articles. **Web scraping** is the craft of extracting that data anyway: fetch the page, parse the markup, and pull out the pieces you want — *politely, legally, and without your pipeline breaking every time a designer moves a `<div>`.*

This is the **first lesson of the Web Scraping module**. It teaches the durable fundamentals — the rules of the road, the HTML tree, and **BeautifulSoup** — that every scraper is built on, whether you hand-roll it or hand the hard part to a managed service later (`15_scraping_with_firecrawl.ipynb`).

> 🧪 **Everything here runs 100% offline.** A real scraper starts with `requests.get(url)` over the network. To keep this notebook deterministic and network-free, we ship a **tiny mock website as Python strings** — a fictional bookshop with a catalogue that spans several pages, detail pages, and a "next page" link — and parse *that*. Every technique is identical to the real thing; only the source of the HTML changes. We show the `requests` call as a comment wherever it belongs.

> 🧭 **The mental model for the whole notebook.** *A web page is a **tree** — the DOM (Document Object Model).* Tags nest inside tags (`<html>` → `<body>` → `<ul>` → `<li>`), each carrying attributes and text. **Scraping is two moves:** (1) **fetch** the tree (an HTTP GET), then (2) **navigate** to the nodes you want and read their text or attributes. BeautifulSoup is your map-and-compass for that tree. And throughout, remember you are a **guest on someone else's server** — knock politely, don't kick the door in.

## ✅ Prerequisites

- **NB 12 (APIs & HTTP)** — status codes, headers, timeouts, retry/backoff. A scrape *is* an HTTP request; we build directly on it.
- **NB 7 (pandas)** — helpful: the payoff of a scrape is usually a tidy **DataFrame**, and §6–§7 land there.
- Comfort with Python lists, dicts, and `for` loops.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Decide **whether** to scrape at all — and scrape **within the rules** (`robots.txt`, Terms of Service, rate limits, copyright, PII).
2. Recall the **HTTP** a scraper needs — status codes, request headers, and an honest `User-Agent`.
3. Read HTML as a **DOM tree** and parse it with `BeautifulSoup(html, "html.parser")` — the standard-library parser, no `lxml` required.
4. Extract data with **`find` / `find_all`** and with **CSS selectors** (`.select` / `.select_one`), reading tag text and attributes.
5. Turn an HTML **table into a pandas DataFrame**, and **follow pagination** across multiple pages.
6. Write a **polite** scraper: throttle with `time.sleep`, cache responses, and honour `robots.txt` with `urllib.robotparser`.
7. Recognise where DIY scraping **breaks** (JavaScript, anti-bot) and what to reach for next.

## 1. What scraping is — and the rules you don't break

Scraping is a **last resort**, not a first move. Before you write a single selector, walk down this list and **stop at the first "yes":**

1. **Is there an official API or data export?** → Use it. (That is NB 12 — stabler, faster, sanctioned. This module's own `16_openalex_scholarly_data.ipynb` is a perfect example: scholarly data via a clean API instead of scraping journal pages.)
2. **Is there a bulk dataset, an RSS feed, or a sitemap?** → Use that.
3. **Only then** consider scraping the rendered HTML.

Once you *do* scrape, you are a **guest on someone else's server**. These are the rules you don't break:

| Rule | What it means in practice |
|---|---|
| **Respect `robots.txt`** | Read `site.com/robots.txt`; never fetch `Disallow`-ed paths; honour `Crawl-delay`. (§8) |
| **Read the Terms of Service** | Some sites *contractually* forbid scraping. **Public ≠ free-to-take.** |
| **Rate-limit yourself** | A human clicks every few seconds; don't fire 100 requests/second. Add delays. (§8) |
| **Identify yourself** | Send a real `User-Agent` with a contact URL. Never spoof one to evade a block. (§2) |
| **Don't take personal data** | Names, emails, faces → **GDPR / CCPA** territory. No PII without a lawful basis. |
| **Respect copyright** | Facts aren't copyrightable; creative content is. Don't republish wholesale. |
| **Cache, don't re-fetch** | Store what you pull so you never request the same page twice. (§8) |

> ⚖️ **Not legal advice.** Scraping law varies by country and is still evolving (e.g. *hiQ v. LinkedIn*). Public data scraped politely is generally lower-risk; bypassing logins, ignoring the ToS, or collecting PII is higher-risk. When in doubt, ask — or use an official API.

> 🧠 **The four-step pipeline.** Every scraper — hand-rolled or managed — is the same shape: **FETCH** (HTTP GET) → **PARSE** (HTML → tree) → **EXTRACT** (pick the nodes) → **STORE** (rows / JSON). You already know *fetch* from NB 12. This notebook is about **parse + extract**.

## 2. HTTP recap for scrapers

A scrape *is* an HTTP request — the same envelope from **NB 12**. Three parts matter most when scraping:

- **Status code** — the server's one-word verdict. `200 OK` you can parse; `403/401` means blocked; `404` gone; **`429 Too Many Requests`** means *slow down* (back off — §8); `5xx` is the server's fault (retry politely).
- **Request headers** — especially **`User-Agent`**, which announces *who* is knocking. A polite scraper sends an honest one with a contact URL, so an admin can email you instead of just banning you.
- **Response headers** — `Content-Type` (is it even HTML?) and rate-limit hints like `Retry-After`.

> ⚠️ **`requests.get()` returns the raw HTML the server sent — nothing more.** It does **not** run JavaScript. If a page builds its content in the browser with JS, `requests` sees an empty shell. That limit defines where DIY scraping ends — see §9.

In [1]:
# Setup — everything below runs OFFLINE. `requests` is imported only so we can
# show the real call in comments; we never touch the network (we parse inline
# HTML fixtures instead).
import re, time, requests
import pandas as pd
from bs4 import BeautifulSoup        # the parser; pip install beautifulsoup4  (imports as bs4)
from urllib import robotparser       # standard-library robots.txt reader (§8)
from urllib.parse import urljoin     # turn relative links into absolute ones (used later)

BASE = "https://books.meridian.test"                       # our fictional bookshop's origin
UA = "MeridianCourseBot/1.0 (+https://example.com/botinfo)"  # an honest, contactable identity
HEADERS = {"User-Agent": UA}

# In production the fetch is one line (see NB 12 for timeouts, retries, raise_for_status):
#     html = requests.get(BASE + "/catalogue/page-1.html", headers=HEADERS, timeout=10).text
# Here fetch() (defined in §3) returns that same HTML from an in-memory fixture instead.

def status_meaning(code):
    '''The first digit of an HTTP status code IS its meaning — a scraper's traffic light.'''
    return {2: "OK        -> parse it",
            3: "redirect  -> follow Location",
            4: "client err-> 403 blocked / 404 gone / 429 slow down",
            5: "server err-> back off and retry (see section 8)"}[code // 100]

for code in (200, 301, 403, 404, 429, 503):
    print(f"  {code}  {status_meaning(code)}")

  200  OK        -> parse it
  301  redirect  -> follow Location
  403  client err-> 403 blocked / 404 gone / 429 slow down
  404  client err-> 403 blocked / 404 gone / 429 slow down
  429  client err-> 403 blocked / 404 gone / 429 slow down
  503  server err-> back off and retry (see section 8)


## 3. The HTML / DOM tree

HTML is not flat text — it is a **tree of nested tags**, the **DOM**. One node contains others:

```text
html
└─ body
   └─ ul#catalogue
      ├─ li.book  (data-genre="scifi")     ← a "node": a tag …
      │  ├─ h3.title  →  "Dune"            ← … with child nodes,
      │  ├─ p.author  →  "Frank Herbert"
      │  └─ p.price   →  "£9.99"           ← … text,
      └─ li.book  (data-genre="fantasy")   ← … and attributes.
```

Three things live on every node, and scraping is just reading them:

| Part | HTML | You read it with |
|---|---|---|
| **Tag name** | `<li> … </li>` | `tag.name` |
| **Attributes** | `class="book"`, `href="…"`, `data-genre="scifi"` | `tag["href"]`, `tag.get("data-genre")` |
| **Text** | the words between the tags | `tag.get_text(strip=True)` |

Let's build our offline mock bookshop — a few catalogue pages, two detail pages, a bestsellers table, and a `robots.txt` — all as plain Python strings.

In [2]:
# ── Our offline mock website: "Meridian Books" ───────────────────────────
# Each value is exactly what requests.get(BASE + path).text would return for
# that page. Parsing these strings is identical to parsing a live response.

_PAGE1 = '''
<!DOCTYPE html>
<html lang="en">
<head><title>Meridian Books - Catalogue (page 1)</title></head>
<body>
  <header><a id="logo" href="/index.html">Meridian Books</a></header>
  <main>
    <h1>Catalogue - page 1 of 3</h1>
    <ul id="catalogue">
      <li class="book" data-genre="scifi">
        <h3 class="title"><a class="book-link" href="/book/dune.html">Dune</a></h3>
        <p class="author">Frank Herbert</p>
        <p class="price">&pound;9.99</p>
        <p class="rating" data-stars="5">*****</p>
        <span class="availability in-stock">In stock (14)</span>
      </li>
      <li class="book" data-genre="fantasy">
        <h3 class="title"><a class="book-link" href="/book/the-hobbit.html">The Hobbit</a></h3>
        <p class="author">J. R. R. Tolkien</p>
        <p class="price">&pound;7.50</p>
        <p class="rating" data-stars="5">*****</p>
        <span class="availability in-stock">In stock (3)</span>
      </li>
      <li class="book" data-genre="nonfiction">
        <h3 class="title"><a class="book-link" href="/book/sapiens.html">Sapiens</a></h3>
        <p class="author">Yuval Noah Harari</p>
        <p class="price">&pound;12.00</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability out-of-stock">Out of stock</span>
      </li>
    </ul>
    <nav class="pager"><a class="next" href="/catalogue/page-2.html">Next -&gt;</a></nav>
  </main>
  <footer>(c) 2026 Meridian Books &middot; <a href="/about.html">About</a></footer>
</body>
</html>
'''

_PAGE2 = '''
<!DOCTYPE html>
<html lang="en">
<head><title>Meridian Books - Catalogue (page 2)</title></head>
<body>
  <main>
    <h1>Catalogue - page 2 of 3</h1>
    <ul id="catalogue">
      <li class="book" data-genre="scifi">
        <h3 class="title"><a class="book-link" href="/book/neuromancer.html">Neuromancer</a></h3>
        <p class="author">William Gibson</p>
        <p class="price">&pound;8.25</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability in-stock">In stock (7)</span>
      </li>
      <li class="book" data-genre="fantasy">
        <h3 class="title"><a class="book-link" href="/book/name-of-the-wind.html">The Name of the Wind</a></h3>
        <p class="author">Patrick Rothfuss</p>
        <p class="price">&pound;10.80</p>
        <p class="rating" data-stars="5">*****</p>
        <span class="availability in-stock">In stock (5)</span>
      </li>
    </ul>
    <nav class="pager"><a class="next" href="/catalogue/page-3.html">Next -&gt;</a></nav>
  </main>
</body>
</html>
'''

_PAGE3 = '''
<!DOCTYPE html>
<html lang="en">
<head><title>Meridian Books - Catalogue (page 3)</title></head>
<body>
  <main>
    <h1>Catalogue - page 3 of 3</h1>
    <ul id="catalogue">
      <li class="book" data-genre="nonfiction">
        <h3 class="title"><a class="book-link" href="/book/educated.html">Educated</a></h3>
        <p class="author">Tara Westover</p>
        <p class="price">&pound;11.40</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability in-stock">In stock (9)</span>
      </li>
      <li class="book" data-genre="scifi">
        <h3 class="title"><a class="book-link" href="/book/foundation.html">Foundation</a></h3>
        <p class="author">Isaac Asimov</p>
        <p class="price">&pound;6.99</p>
        <p class="rating" data-stars="4">****</p>
        <span class="availability out-of-stock">Out of stock</span>
      </li>
    </ul>
    <nav class="pager"><span class="end">- end of catalogue -</span></nav>
  </main>
</body>
</html>
'''

_DUNE = '''
<!DOCTYPE html>
<html lang="en">
<head><title>Dune - Meridian Books</title></head>
<body>
  <main>
    <h1 class="title">Dune</h1>
    <p class="author">by Frank Herbert</p>
    <table class="meta">
      <tr><th>First published</th><td>1965</td></tr>
      <tr><th>Pages</th><td>412</td></tr>
      <tr><th>ISBN</th><td>978-0-441-17271-9</td></tr>
      <tr><th>Genre</th><td>Science fiction</td></tr>
    </table>
    <div class="description"><p>On the desert planet Arrakis, a feud erupts over the spice that powers galactic travel.</p></div>
    <a class="back" href="/catalogue/page-1.html">Back to catalogue</a>
  </main>
</body>
</html>
'''

_HOBBIT = '''
<!DOCTYPE html>
<html lang="en">
<head><title>The Hobbit - Meridian Books</title></head>
<body>
  <main>
    <h1 class="title">The Hobbit</h1>
    <p class="author">by J. R. R. Tolkien</p>
    <table class="meta">
      <tr><th>First published</th><td>1937</td></tr>
      <tr><th>Pages</th><td>310</td></tr>
      <tr><th>ISBN</th><td>978-0-547-92822-7</td></tr>
      <tr><th>Genre</th><td>Fantasy</td></tr>
    </table>
    <div class="description"><p>A reluctant hobbit journeys to a lonely mountain with a company of dwarves.</p></div>
    <a class="back" href="/catalogue/page-1.html">Back to catalogue</a>
  </main>
</body>
</html>
'''

# A standalone HTML table for §6 (this month's bestsellers).
BESTSELLERS_HTML = '''
<table id="bestsellers">
  <thead>
    <tr><th>Rank</th><th>Title</th><th>Author</th><th>Copies sold</th></tr>
  </thead>
  <tbody>
    <tr><td>1</td><td>Dune</td><td>Frank Herbert</td><td>12000</td></tr>
    <tr><td>2</td><td>The Hobbit</td><td>J. R. R. Tolkien</td><td>9800</td></tr>
    <tr><td>3</td><td>Sapiens</td><td>Yuval Noah Harari</td><td>8700</td></tr>
    <tr><td>4</td><td>Neuromancer</td><td>William Gibson</td><td>5400</td></tr>
  </tbody>
</table>
'''

# The site's posted house rules for bots (used in §8).
ROBOTS = '''
User-agent: *
Disallow: /cart/
Disallow: /checkout/
Disallow: /account/
Crawl-delay: 1
Allow: /

User-agent: GreedyBot
Disallow: /
'''

# The whole "site": a path -> HTML map. fetch() is our offline requests.get.
PAGES = {
    "/catalogue/page-1.html": _PAGE1,
    "/catalogue/page-2.html": _PAGE2,
    "/catalogue/page-3.html": _PAGE3,
    "/book/dune.html": _DUNE,
    "/book/the-hobbit.html": _HOBBIT,
}

def fetch(path):
    '''Offline stand-in for requests.get(BASE + path, headers=HEADERS, timeout=10).text'''
    if path not in PAGES:
        raise KeyError("404 Not Found: " + path)   # a real GET would return status 404
    return PAGES[path]

print("Mock site ready:", len(PAGES), "pages")
for _p in PAGES:
    print("   ", _p)

Mock site ready: 5 pages
    /catalogue/page-1.html
    /catalogue/page-2.html
    /catalogue/page-3.html
    /book/dune.html
    /book/the-hobbit.html


In [3]:
# PARSE — turn the raw HTML string into a navigable tree. "html.parser" is built
# into Python (no lxml needed); pass "lxml" instead only if you have installed it.
soup = BeautifulSoup(fetch("/catalogue/page-1.html"), "html.parser")

print("Page <title> tag:", soup.title)                     # a whole <title> node
print("Title text      :", soup.title.get_text(strip=True))
print("The <h1> heading:", soup.h1.get_text(strip=True))
print()

first = soup.find("li", class_="book")                     # first <li class="book"> node
print("First book node - its direct children (tag -> text):")
for child in first.find_all(recursive=False):              # direct child tags only
    print("   ", child.name, "->", child.get_text(strip=True)[:22])
print()

print("Navigate from that node:")
print("   .name          :", first.name)
print("   ['data-genre'] :", first["data-genre"])          # read an attribute
print("   .parent.name   :", first.parent.name, "(the <ul id=catalogue>)")

Page <title> tag: <title>Meridian Books - Catalogue (page 1)</title>
Title text      : Meridian Books - Catalogue (page 1)
The <h1> heading: Catalogue - page 1 of 3

First book node - its direct children (tag -> text):
    h3 -> Dune
    p -> Frank Herbert
    p -> £9.99
    p -> *****
    span -> In stock (14)

Navigate from that node:
   .name          : li
   ['data-genre'] : scifi
   .parent.name   : ul (the <ul id=catalogue>)


## 4. BeautifulSoup basics — `find` and `find_all`

Two methods do most of the work. **`find`** returns the **first** matching node (or `None`); **`find_all`** returns a **list** of every match. Filter by tag name, by `class_=`, by `id=`, or by any attribute.

| Goal | Code |
|---|---|
| First `<h1>` | `soup.find("h1")` |
| First `<li class="book">` | `soup.find("li", class_="book")` |
| **All** `<li class="book">` | `soup.find_all("li", class_="book")` |
| By id | `soup.find(id="catalogue")` |
| By any attribute | `soup.find_all("li", attrs={"data-genre": "scifi"})` |
| The node's text | `node.get_text(strip=True)` |
| An attribute value | `node["href"]` or `node.get("href")` |

> 🔬 **`["href"]` vs `.get("href")`.** `node["href"]` raises `KeyError` if the attribute is missing; `node.get("href")` returns `None` instead. Same choice as with dicts — use `.get()` when the attribute might not be there.

In [4]:
# find_all -> a list of every book node; loop it and read text + attributes.
book_nodes = soup.find_all("li", class_="book")
print(len(book_nodes), "books on page 1:")
print()
for li in book_nodes:
    title  = li.find("h3", class_="title").get_text(strip=True)
    author = li.find("p", class_="author").get_text(strip=True)
    price  = li.find("p", class_="price").get_text(strip=True)
    genre  = li["data-genre"]                              # attribute access
    link   = li.find("a", class_="book-link").get("href")  # .get -> None if missing
    print(f"  {title:22} {price:>7}  [{genre:10}]  {author:18}  -> {link}")

3 books on page 1:

  Dune                     £9.99  [scifi     ]  Frank Herbert       -> /book/dune.html
  The Hobbit               £7.50  [fantasy   ]  J. R. R. Tolkien    -> /book/the-hobbit.html
  Sapiens                 £12.00  [nonfiction]  Yuval Noah Harari   -> /book/sapiens.html


---

### ✋ Quick exercise (~2 min) — Read one book off the page

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using the existing `soup` from §3 (no re-fetching), grab the **first** book node with `soup.find("li", class_="book")`, then pull its **title** text and its **rating stars** (the `data-stars` attribute on the `<p class="rating">`). Print them.

In [5]:
# ✍️ Your turn 👇
# Use the existing `soup` from §3 — no re-fetching.
first_book = soup.find("li", class_="book")
title = ...      # the book's title text (inside <h3 class="title">)
stars = ...      # the data-stars attribute on <p class="rating">  (comes back a string)
# print(title, "-", stars, "stars")

<details>
<summary>✅ <b>Solution</b></summary>

```python
first_book = soup.find("li", class_="book")
title = first_book.find("h3", class_="title").get_text(strip=True)
stars = first_book.find("p", class_="rating")["data-stars"]
print(title, "-", stars, "stars")
```

`find` returns the first matching node; `.get_text(strip=True)` reads its text, and `[...]` reads an attribute. Attributes always come back as **strings** — wrap in `int(stars)` if you need a number.
</details>

## 5. CSS selectors with `.select()`

`find`/`find_all` are fine, but **CSS selectors** are often shorter and more expressive — the very same selectors you'd use in a browser's dev tools or in a stylesheet. Two methods:

- **`soup.select("…")`** → a **list** of every match (like `find_all`).
- **`soup.select_one("…")`** → the **first** match (like `find`).

The selector mini-language:

| Selector | Matches |
|---|---|
| `li` | every `<li>` tag |
| `.book` | every element with `class="book"` |
| `#catalogue` | the element with `id="catalogue"` |
| `li.book` | `<li>` that *also* has class `book` |
| `#catalogue li.book` | `li.book` **anywhere inside** `#catalogue` (descendant) |
| `ul > li` | `<li>` that is a **direct child** of a `<ul>` |
| `a.book-link` | `<a>` with class `book-link` |
| `[data-genre="scifi"]` | any element whose `data-genre` attribute equals `scifi` |

Let's use selectors to extract **every** book on page 1 into a tidy list of dicts — then a DataFrame.

In [6]:
# A reusable extractor: one <li class="book"> node -> a clean dict of fields.
def parse_books(page_soup):
    rows = []
    for li in page_soup.select("li.book"):                      # every <li class="book">
        avail = li.select_one(".availability").get_text(strip=True)
        rows.append({
            "title":    li.select_one(".title").get_text(strip=True),
            "author":   li.select_one(".author").get_text(strip=True),
            "price":    float(re.sub("[^0-9.]", "", li.select_one(".price").get_text())),
            "stars":    int(li.select_one(".rating")["data-stars"]),   # read an attribute
            "genre":    li["data-genre"],
            "in_stock": "In stock" in avail,
            "url":      li.select_one("a.book-link")["href"],
        })
    return rows

page1_df = pd.DataFrame(parse_books(soup))
print(page1_df.to_string(index=False))
print()
print("scifi titles via attribute selector:",
      [a.get_text(strip=True) for a in soup.select('li[data-genre="scifi"] .title')])

     title            author  price  stars      genre  in_stock                   url
      Dune     Frank Herbert   9.99      5      scifi      True       /book/dune.html
The Hobbit  J. R. R. Tolkien   7.50      5    fantasy      True /book/the-hobbit.html
   Sapiens Yuval Noah Harari  12.00      4 nonfiction     False    /book/sapiens.html

scifi titles via attribute selector: ['Dune']


---

### ✋ Quick exercise (~2 min) — Select with CSS

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using `soup` (page 1) and **`.select()`**, build a list of every **author name** on the page. Then, with a single CSS selector, count how many books are **in stock** (hint: those `<span>`s carry the class `in-stock`).

In [7]:
# ✍️ Your turn 👇
authors = ...        # list of author strings, via soup.select(".author")
n_in_stock = ...     # count of <span class="... in-stock">, via soup.select(".in-stock")
# print(authors); print(n_in_stock, "in stock")

<details>
<summary>✅ <b>Solution</b></summary>

```python
authors = [a.get_text(strip=True) for a in soup.select(".author")]
n_in_stock = len(soup.select(".in-stock"))
print(authors)
print(n_in_stock, "in stock")
```

`.select(".author")` returns *all* matching nodes as a list — a comprehension turns them into strings. `.in-stock` selects by class and `len(...)` counts the matches. Selecting a class that *marks the state you want* (here, availability) is sturdier than parsing free text.
</details>

## 6. Extracting a table into pandas

An HTML table is its own little tree: `<table>` → `<thead>`/`<tbody>` → `<tr>` (rows) → `<th>` (headers) / `<td>` (cells). Walk it once and you have a DataFrame.

> 💡 **The one-liner you'll reach for in production:** `pd.read_html(html)` parses *every* `<table>` on a page into a list of DataFrames. It needs a parser backend (`lxml` or `html5lib`) installed — which this offline environment doesn't have — so below we do it **by hand with BeautifulSoup** (no extra dependency, and it shows you exactly what `read_html` does under the hood).

In [8]:
# Parse an HTML <table> into a DataFrame by hand (what pd.read_html does for you).
tbl = BeautifulSoup(BESTSELLERS_HTML, "html.parser").select_one("table#bestsellers")

col_names = [th.get_text(strip=True) for th in tbl.select("thead th")]
rows = []
for tr in tbl.select("tbody tr"):
    rows.append([td.get_text(strip=True) for td in tr.select("td")])

bestsellers_df = pd.DataFrame(rows, columns=col_names)
bestsellers_df["Copies sold"] = bestsellers_df["Copies sold"].astype(int)  # cells arrive as str
print(bestsellers_df.to_string(index=False))
print()
print("Total copies sold:", int(bestsellers_df["Copies sold"].sum()))

# In production the same result is one line (given an lxml / html5lib backend):
#     bestsellers_df = pd.read_html(BESTSELLERS_HTML)[0]

Rank       Title            Author  Copies sold
   1        Dune     Frank Herbert        12000
   2  The Hobbit  J. R. R. Tolkien         9800
   3     Sapiens Yuval Noah Harari         8700
   4 Neuromancer    William Gibson         5400

Total copies sold: 35900


## 7. Following pagination

Real catalogues span many pages, linked by a **"next"** button. The pattern never changes: parse a page, extract its rows, look for the next link, and repeat until there isn't one.

```text
page-1  --(a.next)-->  page-2  --(a.next)-->  page-3  --(no a.next)-->  stop
```

> ⚠️ **Always cap the loop.** A `max_pages` safety net stops a broken "next" link (say, one that points back to itself) from looping forever. Never write an unbounded `while` around a network call.

In [9]:
def crawl_catalogue(start="/catalogue/page-1.html", max_pages=10):
    '''Follow a.next links from `start`, collecting books from every page.'''
    all_rows, path, seen = [], start, set()
    for _ in range(max_pages):                       # cap: never loop forever
        if path is None or path in seen:
            break
        seen.add(path)
        page = BeautifulSoup(fetch(path), "html.parser")   # prod: fetch = requests.get(...).text
        all_rows.extend(parse_books(page))
        nxt = page.select_one("a.next")              # the "next" link, or None on the last page
        path = nxt["href"] if nxt else None
    return pd.DataFrame(all_rows)

catalogue_df = crawl_catalogue()
print(f"Crawled {len(catalogue_df)} books across all pages:")
print(catalogue_df.to_string(index=False))
print()
cheapest = catalogue_df[catalogue_df["in_stock"]].sort_values("price").iloc[0]["title"]
print("Cheapest in-stock book:", cheapest)

Crawled 7 books across all pages:
               title            author  price  stars      genre  in_stock                         url
                Dune     Frank Herbert   9.99      5      scifi      True             /book/dune.html
          The Hobbit  J. R. R. Tolkien   7.50      5    fantasy      True       /book/the-hobbit.html
             Sapiens Yuval Noah Harari  12.00      4 nonfiction     False          /book/sapiens.html
         Neuromancer    William Gibson   8.25      4      scifi      True      /book/neuromancer.html
The Name of the Wind  Patrick Rothfuss  10.80      5    fantasy      True /book/name-of-the-wind.html
            Educated     Tara Westover  11.40      4 nonfiction      True         /book/educated.html
          Foundation      Isaac Asimov   6.99      4      scifi     False       /book/foundation.html

Cheapest in-stock book: The Hobbit


---

### ✋ Quick exercise (~2 min) — Walk the pager

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Forget the books for a moment — just trace the **path** a crawler takes. Starting at `/catalogue/page-1.html`, use `fetch` and `BeautifulSoup` to follow each `a.next` link and collect the list of page paths you visit, in order. Print the list (it should be the three catalogue pages).

In [10]:
# ✍️ Your turn 👇  (reuse `fetch` and BeautifulSoup from above)
path = "/catalogue/page-1.html"
visited = []
# while there is a page to visit: record it, parse it, find a.next, then move on
...
# print(visited)

Ellipsis

<details>
<summary>✅ <b>Solution</b></summary>

```python
path = "/catalogue/page-1.html"
visited = []
while path:
    visited.append(path)
    page = BeautifulSoup(fetch(path), "html.parser")
    nxt = page.select_one("a.next")
    path = nxt["href"] if nxt else None
print(visited)
```

The loop stops naturally when a page has no `a.next` (the last page), setting `path = None`. This *is* pagination — the same three-line move (parse, find next, follow) that `crawl_catalogue` wraps up. In production add a `max_pages` cap so a self-referential "next" link can't loop forever.
</details>

## 8. Polite scraping — four habits

A well-behaved scraper does four things, every time. Two you met in NB 12 (timeouts, retries); the scraping-specific ones are **throttling** and **caching**, plus the **`robots.txt`** check.

1. **Identify** — send an honest `User-Agent` with a contact URL (our `HEADERS`, §2).
2. **Throttle** — `time.sleep(delay)` between requests; honour any `Crawl-delay`. Never fire unbounded parallel hits.
3. **Cache** — store every response so a re-run never re-fetches the same URL. Kind to the server, fast for you.
4. **Obey `robots.txt`** — check *before* you knock. Python's standard library reads and enforces it for you.

First, the robots check with `urllib.robotparser`.

In [11]:
# urllib.robotparser is standard library: it parses robots.txt and answers "may I?"
rp = robotparser.RobotFileParser()
rp.parse(ROBOTS.splitlines())          # in production: rp.set_url(BASE + "/robots.txt"); rp.read()

for p in ["/catalogue/page-2.html", "/cart/checkout", "/account/settings", "/book/dune.html"]:
    ok = rp.can_fetch(UA, BASE + p)
    print(f"  {'allowed' if ok else 'BLOCKED'}  {p}")

print()
print("Crawl-delay requested:", rp.crawl_delay(UA), "second(s) -> sleep at least this long")
print("GreedyBot may fetch '/':", rp.can_fetch("GreedyBot", BASE + "/"), "(banned outright)")

  allowed  /catalogue/page-2.html
  BLOCKED  /cart/checkout
  BLOCKED  /account/settings
  allowed  /book/dune.html

Crawl-delay requested: 1 second(s) -> sleep at least this long
GreedyBot may fetch '/': False (banned outright)


In [12]:
# A tiny polite fetcher: THROTTLE + CACHE + robots-awareness, with the network
# injected so it runs offline. In production you pass a real requests-based fetch.
class PoliteScraper:
    def __init__(self, fetch_fn, robots, user_agent=UA, default_delay=0.2):
        self.fetch_fn = fetch_fn        # (path) -> html ; inject requests in production
        self.robots = robots            # a parsed RobotFileParser
        self.user_agent = user_agent
        self.delay = robots.crawl_delay(user_agent) or default_delay   # honour Crawl-delay
        self._cache = {}                # path -> html ; a real one might persist to disk
        self._last = 0.0

    def get(self, path):
        if not self.robots.can_fetch(self.user_agent, BASE + path):
            print(f"  robots-skip  {path}")          # OBEY: never fetch a disallowed path
            return None
        if path in self._cache:
            print(f"  cache        {path}")          # CACHE: never re-fetch
            return self._cache[path]
        wait = self.delay - (time.monotonic() - self._last)
        if wait > 0:
            time.sleep(wait)                          # THROTTLE: respect the gap
        html = self.fetch_fn(path)                    # (would carry HEADERS in production)
        self._last = time.monotonic()
        self._cache[path] = html
        print(f"  FETCH        {path}")
        return html

# A real requests session would look like this (offline here, so we do not call it):
#     session = requests.Session(); session.headers.update(HEADERS)
#     scraper = PoliteScraper(lambda p: session.get(BASE + p, timeout=10).text, rp)
scraper = PoliteScraper(fetch, rp)

t0 = time.monotonic()
scraper.get("/catalogue/page-1.html")   # FETCH
scraper.get("/catalogue/page-2.html")   # FETCH (throttled by Crawl-delay)
scraper.get("/catalogue/page-1.html")   # cache hit -> instant, no network
scraper.get("/account/settings")        # blocked by robots.txt -> skipped
print()
print(f"elapsed {time.monotonic() - t0:.2f}s (throttle + cache + robots all enforced)")

  FETCH        /catalogue/page-1.html


  FETCH        /catalogue/page-2.html
  cache        /catalogue/page-1.html
  robots-skip  /account/settings

elapsed 1.01s (throttle + cache + robots all enforced)


---

### ✋ Quick exercise (~2 min) — Ask robots first

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A polite crawler checks the rules *before* it knocks. Reusing `rp` (§8) and `UA`, loop over the paths below and print, for each, whether a bot is allowed to fetch it. How many of the four are allowed?

```python
paths = ["/catalogue/page-3.html", "/cart/", "/book/the-hobbit.html", "/account/orders"]
```

In [13]:
# ✍️ Your turn 👇  (reuse `rp` and `UA` from §8)
paths = ["/catalogue/page-3.html", "/cart/", "/book/the-hobbit.html", "/account/orders"]
allowed = ...      # count how many of `paths` rp.can_fetch(UA, BASE + path) allows
# for p in paths:
#     print(p, rp.can_fetch(UA, BASE + p))

<details>
<summary>✅ <b>Solution</b></summary>

```python
paths = ["/catalogue/page-3.html", "/cart/", "/book/the-hobbit.html", "/account/orders"]
allowed = 0
for p in paths:
    ok = rp.can_fetch(UA, BASE + p)
    print(f"  {'allowed' if ok else 'BLOCKED'}  {p}")
    allowed += ok
print(allowed, "of", len(paths), "allowed")
```

`/cart/` and `/account/orders` sit under `Disallow` rules, so `can_fetch` returns `False`; the catalogue and book paths are allowed → **2 of 4**. Checking `can_fetch` *before* fetching means a disallowed URL never even enters your queue. (`allowed += ok` works because `True`/`False` count as `1`/`0`.)
</details>

## 9. ⚠️ Where DIY scraping breaks

`requests` + BeautifulSoup is perfect for **static, server-rendered HTML** — like every page in this notebook. Modern sites are often neither static nor simple. The pain points:

| Problem | Why `requests` + `bs4` struggles | The usual fix |
|---|---|---|
| **JavaScript rendering** | `requests` gets the empty shell; content is drawn later by JS in a browser | a **headless browser**: Playwright or Selenium |
| **Anti-bot / Cloudflare** | challenges, fingerprinting, IP blocks | rotating proxies, stealth browsers, or a managed service |
| **Layout drift** | selectors break on every redesign — you get `None`, then an `AttributeError` | resilient selectors, monitoring, or LLM extraction |
| **Login / sessions** | content behind auth, CSRF tokens, cookies | an authenticated `requests.Session` — carefully, and within the ToS |
| **"Just give me clean text for my LLM"** | HTML is full of nav, ads, and boilerplate noise | a service that returns clean **markdown** |

> 🧭 **Forward pointers.** When a page needs a real browser, reach for a **headless browser** — **Playwright** or **Selenium** (named here only; they drive a real Chromium so the JavaScript actually runs). When you'd rather hand the whole mess — JS, anti-bot, boilerplate — to an API that returns clean, LLM-ready markdown or typed JSON, that's the next lesson: **`15_scraping_with_firecrawl.ipynb`**. And remember §1's ladder: if the data has an **API**, prefer it — e.g. **`16_openalex_scholarly_data.ipynb`** pulls scholarly records from a clean API instead of scraping.

> ⚠️ **Guard against silent breakage.** `select_one(".price")` returns `None` the moment that class is renamed; `None.get_text()` then raises `AttributeError`. Defend yourself: prefer **stable hooks** (ids, `data-*` attributes, semantic tags) over generated class names, **guard every lookup** for `None`, and **alert on zero rows** — "found 0 books" almost always means the layout moved, not that the shop is empty. You'll reproduce exactly this failure in Exercise 5.

## 🧪 Practice exercises

These reuse the fixtures and helpers built above (`soup`, `fetch`, `parse_books`, `catalogue_df`, `BESTSELLERS_HTML`). Try each before opening the solution.

### Exercise 1 — ⭐ Titles and prices

From page 1's `soup`, build a list of `(title, price_text)` tuples — one per book — using either `find_all` or `.select`. Print them.

In [14]:
# Your code here  👇
pairs = ...
# for t, p in pairs:
#     print(f"{t:22} {p}")

<details>
<summary>💡 <b>Solution</b></summary>

```python
pairs = [(li.select_one(".title").get_text(strip=True),
          li.select_one(".price").get_text(strip=True))
         for li in soup.select("li.book")]
for t, p in pairs:
    print(f"{t:22} {p}")
```

One comprehension over `soup.select("li.book")` pulls both fields per node. Keeping the price as *text* (not `float`) preserves the currency symbol — parse to a number only when you need to compute.
</details>

### Exercise 2 — ⭐⭐ Parse a detail page

Fetch the Dune detail page (`fetch("/book/dune.html")`), parse it, and extract its metadata `<table class="meta">` into a **dict** like `{"First published": "1965", "Pages": "412", ...}`. Each row is a `<tr>` holding a `<th>` (key) and a `<td>` (value).

In [15]:
# Your code here  👇
detail = BeautifulSoup(fetch("/book/dune.html"), "html.parser")
meta = ...
# print(meta)

<details>
<summary>💡 <b>Solution</b></summary>

```python
detail = BeautifulSoup(fetch("/book/dune.html"), "html.parser")
meta = {}
for tr in detail.select("table.meta tr"):
    key = tr.find("th").get_text(strip=True)
    val = tr.find("td").get_text(strip=True)
    meta[key] = val
print(meta)
```

Each `<tr>` pairs one `<th>` (label) with one `<td>` (value) — reading them row by row rebuilds the record as a dict. That dict-per-item shape is exactly what you'd write to a database row.
</details>

### Exercise 3 — ⭐⭐ Filter the catalogue

Using `catalogue_df` from §7, select the **in-stock** books, sort them by **price** ascending, and show just the `title`, `author`, and `price` columns. (This is the pandas payoff of a scrape — NB 7.)

In [16]:
# Your code here  👇
result = ...
# print(result.to_string(index=False))

<details>
<summary>💡 <b>Solution</b></summary>

```python
result = (catalogue_df[catalogue_df["in_stock"]]
          .sort_values("price")[["title", "author", "price"]])
print(result.to_string(index=False))
```

`catalogue_df["in_stock"]` is a boolean mask; `sort_values("price")` orders the survivors; the double-bracket list picks the columns. Once a scrape lands in a DataFrame, every pandas trick from NB 7 is on the table.
</details>

### Exercise 4 — ⭐⭐ Average rating per genre

Using `catalogue_df`, compute the **average `stars`** for each `genre`, sorted highest first. (Hint: `groupby`.)

In [17]:
# Your code here  👇
by_genre = ...
# print(by_genre)

<details>
<summary>💡 <b>Solution</b></summary>

```python
by_genre = (catalogue_df.groupby("genre")["stars"]
            .mean()
            .sort_values(ascending=False))
print(by_genre)
```

`groupby("genre")["stars"].mean()` collapses each genre to its average rating. Grouping a scraped DataFrame is where raw HTML finally turns into an *insight*.
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

The helper below is meant to read each book's **discount badge**, but it **crashes**. Run it, read the traceback, work out the root cause, and fix it so it returns a sensible default when there is no discount.

In [18]:
# 👇 Buggy on purpose — run it, see the error, then write your fix below.
def get_discount(book_li):
    # our book cards have NO <span class="discount">, so .find(...) returns None
    return book_li.find("span", class_="discount").text

first_book = soup.find("li", class_="book")
print(get_discount(first_book))     # AttributeError: 'NoneType' object has no attribute 'text'

AttributeError: 'NoneType' object has no attribute 'text'

<details>
<summary>💡 <b>Solution</b></summary>

**Root cause.** `book_li.find("span", class_="discount")` finds nothing (no book has a discount badge), so it returns `None`. Calling `.text` on `None` raises `AttributeError: 'NoneType' object has no attribute 'text'`. This is *the* classic scraping bug — and exactly how a silent layout change surfaces at runtime.

```python
def get_discount(book_li, default="no discount"):
    badge = book_li.find("span", class_="discount")    # may be None
    return badge.get_text(strip=True) if badge else default

first_book = soup.find("li", class_="book")
print(get_discount(first_book))                         # -> "no discount"
```

**Lesson.** Never chain `.text` / `.get_text()` straight onto a `find`/`select_one` result you haven't checked. Capture the node, test it for `None`, *then* read it — the discipline from §9.
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ A resilient row extractor

Write `safe_parse_book(li)` that returns a dict with `title`, `price`, and `stars`, but **never crashes** on a missing field — use `None` (or a default) for anything absent. Test it on a normal book node *and* on a broken one: `BeautifulSoup("<li class='book'></li>", "html.parser").li`.

In [19]:
# Your code here  👇
def safe_parse_book(li):
    ...
# print(safe_parse_book(soup.find("li", class_="book")))
# print(safe_parse_book(BeautifulSoup("<li class='book'></li>", "html.parser").li))

<details>
<summary>💡 <b>Solution</b></summary>

```python
def safe_parse_book(li, default=None):
    def text(sel):
        node = li.select_one(sel)
        return node.get_text(strip=True) if node else default
    rating = li.select_one(".rating")
    return {
        "title": text(".title"),
        "price": text(".price"),
        "stars": int(rating["data-stars"]) if rating and rating.has_attr("data-stars") else default,
    }

print(safe_parse_book(soup.find("li", class_="book")))
print(safe_parse_book(BeautifulSoup("<li class='book'></li>", "html.parser").li))
```

A tiny `text()` helper centralises the `None`-guard so every field is defended the same way. The broken node returns `{"title": None, "price": None, "stars": None}` instead of raising — the scraper survives a layout change *and* you can detect it (all-`None` rows) rather than crash.
</details>

### Stretch exercise B — ⭐⭐⭐ Absolute URLs

The book links are **relative** (`/book/dune.html`). Crawlers need **absolute** URLs. Using `urllib.parse.urljoin` (already imported) and `BASE`, collect every book's absolute detail URL from page 1.

In [20]:
# Your code here  👇
abs_urls = ...
# for u in abs_urls: print(u)

<details>
<summary>💡 <b>Solution</b></summary>

```python
abs_urls = [urljoin(BASE, a["href"]) for a in soup.select("a.book-link")]
for u in abs_urls:
    print(u)
```

`urljoin(BASE, "/book/dune.html")` → `"https://books.meridian.test/book/dune.html"`. `urljoin` also resolves `../` and relative paths correctly — always prefer it over string concatenation, which breaks on the edge cases.
</details>

### Stretch exercise C — ⭐⭐⭐ Cache hits, counted

Extend the idea behind `PoliteScraper`: write a fetcher wrapper that counts **network fetches** vs **cache hits**. Fetch `/catalogue/page-1.html` three times and `/catalogue/page-2.html` once, then report `2 fetches, 2 cache hits`.

In [21]:
# Your code here  👇
# Wrap `fetch` with a dict cache and two counters.
...

Ellipsis

<details>
<summary>💡 <b>Solution</b></summary>

```python
class CountingCache:
    def __init__(self, fetch_fn):
        self.fetch_fn, self.cache = fetch_fn, {}
        self.fetches = self.hits = 0
    def get(self, path):
        if path in self.cache:
            self.hits += 1
            return self.cache[path]
        self.fetches += 1
        self.cache[path] = self.fetch_fn(path)
        return self.cache[path]

c = CountingCache(fetch)
for p in ["/catalogue/page-1.html", "/catalogue/page-1.html",
          "/catalogue/page-1.html", "/catalogue/page-2.html"]:
    c.get(p)
print(f"{c.fetches} fetches, {c.hits} cache hits")
```

Two distinct URLs → **2 fetches**; the two repeat requests for page 1 → **2 cache hits**. A cache is the single biggest politeness win: it turns re-runs into zero server load.
</details>

## 🎁 Bonus mini-project — A polite paginated scraper

Combine everything into one function, `scrape_bookshop()`, that:

1. Checks `robots.txt` before each request (reuse `rp`).
2. Follows pagination from `/catalogue/page-1.html` to the end.
3. Throttles and caches via a `PoliteScraper`.
4. Returns a **clean, typed** pandas DataFrame (prices as `float`, `in_stock` as `bool`), sorted by price.

Then print a one-line health check — *"N books, M in stock, avg £X.XX"* — and **alert if N is 0** (the layout-moved smell from §9).

In [22]:
# Your code here  👇
def scrape_bookshop():
    ...
# df = scrape_bookshop()
# print(df.to_string(index=False))

<details>
<summary>💡 <b>Solution sketch</b></summary>

```python
def scrape_bookshop(start="/catalogue/page-1.html", max_pages=10):
    scraper = PoliteScraper(fetch, rp)                # robots + throttle + cache
    rows, path, seen = [], start, set()
    for _ in range(max_pages):                        # cap the loop (§7)
        if not path or path in seen:
            break
        seen.add(path)
        html = scraper.get(path)                      # None if robots blocks it
        if html is None:
            break
        page = BeautifulSoup(html, "html.parser")
        rows.extend(parse_books(page))
        nxt = page.select_one("a.next")
        path = nxt["href"] if nxt else None
    return pd.DataFrame(rows).sort_values("price").reset_index(drop=True)

df = scrape_bookshop()
n, m = len(df), int(df["in_stock"].sum())
assert n > 0, "0 books scraped — the layout probably moved (see §9)!"
print(df.to_string(index=False))
print(f"{n} books, {m} in stock, avg GBP {df['price'].mean():.2f}")
```

This is the whole module in one function: **obey robots → fetch politely (throttle + cache) → follow pagination → parse → return a typed DataFrame**, with a zero-row assertion as your early-warning system. Swap `fetch` for a real `requests`-based fetcher and the *exact same code* scrapes a live site.
</details>

## 🧠 Key takeaways

- **Scrape only as a last resort.** API > bulk export / RSS > scraping — and always **within the rules**: `robots.txt`, ToS, rate limits, no PII, respect copyright.
- A web page is a **DOM tree**. Scraping is **fetch → parse → extract → store**; this notebook owns *parse + extract*.
- Parse with **`BeautifulSoup(html, "html.parser")`** — the standard-library parser, no `lxml` needed.
- **`find`/`find_all`** and **CSS `.select`/`.select_one`** are the two ways to reach nodes; read text with `get_text(strip=True)` and attributes with `node["attr"]` / `node.get("attr")`.
- Tables become DataFrames by walking `thead`/`tbody`; **pagination** is a `while` loop that follows the `next` link — always with a `max_pages` cap.
- **Be polite:** an honest `User-Agent`, a `time.sleep` throttle, a cache, and a `urllib.robotparser` `can_fetch` check *before* every request.
- **Selectors are brittle.** Guard every lookup for `None`, prefer stable hooks, and alert on zero rows — a redesign breaks scrapers silently (Exercise 5).
- DIY breaks on **JavaScript and anti-bot** pages → a headless browser (Playwright/Selenium) or a managed API (`15_scraping_with_firecrawl.ipynb`).

## ✅ Self-assessment

- [ ] Explain when to scrape vs. use an API, and name three rules you must not break
- [ ] Parse an HTML string into a tree with `BeautifulSoup(html, "html.parser")`
- [ ] Extract text and attributes with both `find`/`find_all` and `.select`/`.select_one`
- [ ] Turn an HTML table into a pandas DataFrame
- [ ] Follow a `next` link across pages, with a loop cap
- [ ] Check `robots.txt` with `urllib.robotparser` before fetching
- [ ] Guard a scraper against a missing element (no `AttributeError` on `None`)

## 🚀 Next step

You can now scrape any **static** page politely. The next lesson tackles the pages that defeat `requests` — JavaScript-heavy, anti-bot, "just give me clean markdown" — by handing the hard part to a managed API: **`15_scraping_with_firecrawl.ipynb`**. And when the data already has a clean API, prefer it — see **`16_openalex_scholarly_data.ipynb`**.